### Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, avg, median, col, coalesce, lit, when

### Read from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info_raw")

df.display()

In [0]:
df.printSchema()

### Silver transformations

Rename columns with proper column names

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_number",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

Display nulls in product_cost, product_line, product_start_date, product_end_date

In [0]:
df.filter(col("product_cost").isNull()).show(truncate=False)

df.filter(col("product_line").isNull()).show(truncate=False)

df.filter(col("product_start_date").isNull()).show(truncate=False)

df.filter(col("product_end_date").isNull()).show(truncate=False)

Remove nulls from product_cost, product_line and product_start_date.

product_end_date is not included as we have yet to know when it will be phased out.

In [0]:
df = df.dropna(subset=["product_cost"])

df = df.dropna(subset=["product_line"])

df = df.dropna(subset=["product_start_date"])

### Write into Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_products
LIMIT 10